# SPL Chatbot — Llama 3 Fine-tuning with QLoRA

Fine-tunes Meta Llama 3 8B on SPL Q&A data using QLoRA.
Runs on Kaggle free T4 GPU (16GB VRAM).

**Steps:**
1. Install dependencies
2. Load dataset
3. Load Llama 3 in 4-bit quantization
4. Apply QLoRA adapters
5. Train
6. Save and upload to Hugging Face

## Step 1 — Install Dependencies

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 bitsandbytes==0.43.1 accelerate==0.29.3 datasets huggingface_hub

## Step 2 — Configuration

In [ ]:
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login

# ── Config ────────────────────────────────────────────────────────────────────
HF_TOKEN        = "hf_your_token_here"          # paste your HF Write token
HF_USERNAME     = "your_hf_username"             # your HuggingFace username
MODEL_ID        = "meta-llama/Meta-Llama-3-8B-Instruct"
OUTPUT_MODEL    = f"{HF_USERNAME}/spl-llama3-8b-qlora"
OUTPUT_DIR      = "/kaggle/working/spl-llama3"

# Training config
MAX_SEQ_LEN     = 512
EPOCHS          = 3
BATCH_SIZE      = 2
GRAD_ACCUM      = 4          # effective batch size = 8
LEARNING_RATE   = 2e-4

# LoRA config
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3 — Login to Hugging Face

In [ ]:
login(token=HF_TOKEN)
print("Logged in to Hugging Face")

## Step 4 — Load and Format Dataset

In [ ]:
# Upload your finetune_dataset_final.json to Kaggle as a dataset
# Then update this path accordingly
DATASET_PATH = "/kaggle/input/spl-finetune-dataset/finetune_dataset_final.json"

with open(DATASET_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} training examples")
print("Sample:")
print(f"Q: {raw_data[0]['instruction']}")
print(f"A: {raw_data[0]['response'][:100]}...")


def format_llama3(example):
    """
    Format into Llama 3 chat template.
    This is the exact format Meta uses for instruction tuning.
    """
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are the AI assistant for the System Performance Laboratory (SPL) "
        "at Virginia Tech. Answer questions accurately and concisely about SPL's "
        "research, team, partnerships, and how to get involved."
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{example['instruction']}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{example['response']}"
        "<|eot_id|>"
    )
    return {"text": text}


# Convert to HuggingFace Dataset and format
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_llama3)

# Split 90% train, 10% validation
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)} examples")
print(f"Eval:  {len(eval_dataset)} examples")

## Step 5 — Load Model in 4-bit Quantization (QLoRA)

In [ ]:
# 4-bit quantization config — this is what makes it fit on T4 16GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 from QLoRA paper
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,      # extra memory saving
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

print(f"Model loaded. Parameters: {model.num_parameters():,}")

## Step 6 — Apply LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,                    # rank of low-rank matrices
    lora_alpha=LORA_ALPHA,       # scaling factor
    lora_dropout=LORA_DROPOUT,
    target_modules=[             # which layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)

# Show how many parameters are actually being trained
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of total)")
# Expected output: ~0.08% of parameters — that's the magic of QLoRA!

## Step 7 — Train

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=False,
    bf16=True,
    report_to="none",
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

print("Starting training...")
print(f"Training on {len(train_dataset)} examples for {EPOCHS} epochs")
print(f"Estimated time: 45-90 minutes on T4 GPU")

trainer.train()
print("Training complete!")

## Step 8 — Save and Upload to Hugging Face

In [ ]:
# Save locally first
print("Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Upload LoRA adapter to Hugging Face (private repo)
print(f"Uploading to Hugging Face as {OUTPUT_MODEL}...")
model.push_to_hub(
    OUTPUT_MODEL,
    token=HF_TOKEN,
    private=True,        # keep it private
)
tokenizer.push_to_hub(
    OUTPUT_MODEL,
    token=HF_TOKEN,
    private=True,
)

print(f"Model uploaded to: https://huggingface.co/{OUTPUT_MODEL}")
print("Fine-tuning complete!")

## Step 9 — Quick Test

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.3,
)

test_questions = [
    "Who leads SPL?",
    "How many PhD students does SPL have?",
    "What are SPL's research areas?",
]

for q in test_questions:
    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are the SPL assistant at Virginia Tech.<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{q}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    result = pipe(prompt)
    answer = result[0]["generated_text"].split("assistant")[-1].strip()
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()